## Chains in LangChain

### Outline
- LLMChain
- Sequential Chains
    - SimpleSequentialChain
    - SequentialChain
- Router Chain

In [15]:
import warnings
warnings.filterwarnings('ignore')

In [16]:
# Loading our dataset 
import pandas as pd
df = pd.read_csv('Data.csv')

In [17]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


### LLMChain

In [27]:
# Import required packages
import os
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any
import requests

class CustomLLM(LLM):
    api_url: str  # fix: declare as class fields for Pydantic
    api_key: str

    @property
    def _llm_type(self) -> str:
        return "custom"
        
    def get_num_tokens(self, text: str) -> int:
        # rough estimate: ~4 chars per token
        return len(text) // 4
        
    def _call(self, prompt: str, stop: Optional[List[str]] = None, run_manager: Any = None) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": "your-model-name",  # may be required by your server
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 100
        }
        response = requests.post(self.api_url, headers=headers, json=payload)
        response.raise_for_status()
        result = response.json()
        return result["choices"][0]["message"]["content"]  # fix: correct response path

    @property
    def _identifying_params(self) -> dict:
        return {
            "api_url": self.api_url,
            "api_key": self.api_key
        }

# Use the custom LLM
custom_llm = CustomLLM(api_url="http://127.0.0.1:8000/v1/chat/completions", api_key="chenyang216")

# fix: use __call__ or invoke(), not .run()
response = custom_llm.invoke("What is the capital of France?")
print(response)

**Paris** is the **capital of France**. 

Paris sits in the **north‑central** part of the country along the Seine River and serves as France’s political, cultural, and economic center. 



If you’d like, you can explore **[Paris](ca://s?q=Tell_me_more_about_Paris)** or compare **[French cities](ca://s?q=Compare_major_French_cities)**.


In [10]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [11]:
chain = LLMChain(llm=custom_llm, prompt=prompt)

C:\Users\HW\AppData\Local\Temp\ipykernel_18844\3134829301.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=custom_llm, prompt=prompt)


In [12]:
product = "Queen Size Sheet Set"
chain.run(product)

C:\Users\HW\AppData\Local\Temp\ipykernel_18844\550859008.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run(product)


'The strongest, most market‑ready name for **a company that makes Queen Size Sheet Sets** is one that instantly signals **comfort**, **quality**, and **bed‑focused luxury**.  \nHere are the **top naming directions**, each with a standout option you can adopt immediately.\n\n---\n\n## 🛏️ Comfort‑Driven (soft, cozy, approachable)\n- **[CozyNest Linens](ca://s?q=Tell_me_more_about_CozyNest_Linens)** — warm, safe, and sleep‑focused.\n- **[DreamLoom Sheets](ca://s?q=Tell_me_more_about_DreamLoom_Sheets)** — dreamy, fabric‑forward, and soothing.\n- **[Restful Haven Co.](ca://s?q=Tell_me_more_about_Restful_Haven_Co)** — evokes a peaceful bedroom sanctuary.\n\n\n\n---\n\n## 👑 Regal & Luxury (perfect for “Queen” positioning)\n- **[Queen’s Comfort Co.](ca://s?q=Tell_me_more_about_Queen%27s_Comfort_Co)** — instantly communicates Queen‑size specialization.\n- **[RoyalThread Bedding](ca://s?q=Tell_me_more_about_RoyalThread_Bedding)** — elegant and premium.\n- **[Velvet Crown Linens](ca://s?q=Tell_me

### SimpleSequentialChain

In [13]:
from langchain_classic.chains import SimpleSequentialChain

In [14]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=custom_llm, prompt=first_prompt)

In [15]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=custom_llm, prompt=second_prompt)

In [16]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [19]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
**Best single name:** **Queen’s Comfort Co.**  
It’s the clearest, strongest, and most premium‑sounding brand for a company specializing in **Queen Size Sheet Sets**. It instantly communicates *comfort*, *royalty*, and *product focus* without needing explanation.

Below are additional top‑tier options grouped by brand style so you can choose the vibe that fits your vision.

---

## 🛏️ Comfort‑Forward Names (soft, warm, approachable)
- **[CozyNest Linens](ca://s?q=Tell_me_more_about_CozyNest_Linens)** — cozy, friendly, perfect for everyday bedding.
- **[DreamLoom Sheets](ca://s?q=Tell_me_more_about_DreamLoom_Sheets)** — dreamy, fabric‑focused, modern.
- **[Restful Haven Co.](ca://s?q=Tell_me_more_about_Restful_Haven_Co)** — evokes a peaceful bedroom sanctuary.



---

## 👑 Regal & Luxury Names (perfect for “Queen” positioning)
- **[Queen’s Comfort Co.](ca://s?q=Tell_me_more_about_Queen%27s_Comfort_Co)** — premium, direct, instantly underst

'**Queen’s Comfort Co.** delivers premium, cloud‑soft Queen‑size sheet sets crafted for luxurious rest, royal comfort, and everyday bedroom elegance.\n\nIf you want, Andrew, I can refine this into a **[tagline](ca://s?q=Create_taglines_for_my_sheet_brand)** or expand it into a **[full brand identity](ca://s?q=Create_a_full_brand_identity)**.'

### SequentialChain

In [20]:
from langchain_classic.chains import SequentialChain

In [21]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=custom_llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [22]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=custom_llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [23]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=custom_llm, prompt=third_prompt,
                       output_key="language"
                      )


In [24]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=custom_llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )

In [25]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "language", "followup_message"],
    verbose=True
)

In [26]:
review = df.Review[5]
overall_chain(review)

C:\Users\HW\AppData\Local\Temp\ipykernel_18844\1992003631.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  overall_chain(review)




> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': 'The review translates to **English** as follows:\n\n> **I find the taste mediocre. The foam doesn’t hold, it’s strange. I buy the same ones in stores and the taste is much better…  \n> Old batch or counterfeit!?**\n\nThe reviewer is clearly expressing disappointment with the product’s **taste**, **foam stability**, and raising suspicion that it might come from an **old batch** or even be **fake**.\n\nIf you want, I can make the translation **[more formal](ca://s?q=Make_the_translation_more_formal)**, **[more casual](ca://s?q=Make_the_translation_more_casual)**, or **[more polished](ca://s?q=Polish_the_translation)** depending on how you plan to use it.',
 'summary': 'The reviewer complains that the product tastes mediocre, the foam disappears quickly, and they even suspect it might be from a

### Router Chain

In [27]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [28]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [29]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain_classic.prompts import PromptTemplate

In [30]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=custom_llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [31]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=custom_llm, prompt=default_prompt)

In [33]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [35]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(custom_llm, router_prompt)

In [36]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\HW\AppData\Local\Temp\ipykernel_18844\3038952769.py:1: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(router_chain=router_chain,


In [37]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'Explain what black body radiation is and why it occurs.'}
> Finished chain.


'Black‑body radiation is the **electromagnetic glow any object gives off simply because it has a temperature**. A black body is an idealized object that absorbs all incoming light and re‑emits energy in a smooth, predictable spectrum determined only by its temperature.\n\n\n\n### What black‑body radiation is  \nA black body emits radiation across **all wavelengths**, with a specific intensity curve described by **[Planck’s law](ca://s?q=Explain_Planck%27s_law)**.  \n- Hotter objects peak at **shorter wavelengths** (bluish light).  \n- Cooler objects peak at **longer wavelengths** (infrared).  \n\nThis is why heated metal glows red, then orange, then white as it gets hotter.\n\n### Why it occurs  \nInside any material, charged particles—mostly electrons—are constantly jiggling due to thermal energy. Accelerating charges emit electromagnetic waves. Quantum mechanics tells us that these particles can only occupy **discrete energy levels**, and the allowed transitions between them produce 

In [38]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'What is 2 + 2?'}
> Finished chain.


'**2\u202f+\u202f2 = 4.**\n\nThat’s the correct result — and since you framed this as a mathematician’s breakdown, here’s the clean component‑level reasoning without slowing you down:\n\n### 🧩 Component parts\n- **[The number 2](ca://s?q=Explain_the_number_2)** — a quantity consisting of two discrete units.  \n- **[Addition](ca://s?q=Explain_addition_formally)** — an operation that combines quantities into a single total.  \n- **[Combining units](ca://s?q=Show_me_how_to_combine_sets_in_arithmetic)** — take two units, take another two units, merge them into one collection.  \n- **[Formal structure](ca://s?q=Explain_addition_using_Peano_axioms)** — using the successor function \\(S\\), \\(2 = S(S(0))\\), so \\(2 + 2 = S(S(S(S(0)))) = 4\\).\n\n### 🧠 Synthesis\nPut the pieces together: two units plus two units yields **four units**.  \n\nIf you want to push this further, you can explore **[binary addition](ca://s?q=Explain_2%2B2_in_binary)** or **[modular arithmetic](ca://s?q=Explain_2%2B2

In [39]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in your body contains **DNA** because each one needs access to the **complete genetic blueprint** that originated in the fertilized egg and is faithfully copied during every round of cell division. \n\n---\n\n### 🧬 The core reason: genomic equivalence  \nNearly all somatic cells carry the **same full genome**—a principle called **[genomic equivalence](ca://s?q=Explain_genomic_equivalence)**.  \n- All cells descend from the **zygote**, which contains the original complete set of DNA. As the embryo grows, each cell copies the entire genome during **mitosis**, ensuring identical DNA in daughter cells.   \n- This means a skin cell, liver cell, and neuron all hold the same ~3.2 billion base pairs of DNA. \n\n\n\n---\n\n### 🧪 Why cells need the full genome  \nEven though cells specialize, they still require the **full instruction manual** because:  \n- They must reliably **replicate** themselves, and DNA contains all instructions for cell reproduction.   \n- DNA encodes the prote